In [ ]:
# =============================================================================
# MODEL 5 - SARIMA (Optimised - full London, auto order selection)
# Task: Predict Crime_Count per LSOA based on historical time series
#
# Strategy:
#   - auto_arima selects best (p,d,q)(P,D,Q,12) per LSOA via AIC
#   - Parallel fitting across all CPU cores (joblib)
#   - Full London LSOA set - no sampling
#   - Refit on full 2020-2023 series, forecast all 12 months of 2024
# =============================================================================

import os
import time
import warnings
import subprocess
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================
DATA_PATH = "./crime_per_capita_df.csv"
MODEL_COMPARISON_PATH = "model_comparison.csv"
PREDICTIONS_PATH = "sarima_predictions.csv"
REPORT_TXT_PATH = "evaluation_results_sarima.txt"
ARTIFACT_PATH = Path("artifacts") / "sarima_model.joblib"

TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEAR = 2024
N_FORECAST = 12
MIN_TRAIN_OBS = 24
N_JOBS = -1

HOTSPOT_PERCENTS = [5, 10, 20]

AUTO_ARIMA_CONFIG = {
    "start_p": 0,
    "max_p": 3,
    "start_q": 0,
    "max_q": 3,
    "d": None,
    "max_d": 2,
    "start_P": 0,
    "max_P": 2,
    "start_Q": 0,
    "max_Q": 2,
    "D": None,
    "max_D": 1,
    "m": 12,
    "seasonal": True,
    "information_criterion": "aic",
    "stepwise": True,
    "suppress_warnings": True,
    "error_action": "ignore",
    "trace": False,
}

FALLBACK_ORDER = (1, 1, 1)
FALLBACK_SEASONAL_ORDER = (1, 1, 1, 12)

# =============================================================================
# 2. HELPERS
# =============================================================================
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator == 0.0:
        return np.nan
    return float(numerator) / denominator


def round_or_nan(value, digits=4):
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), digits)


def fmt4(value):
    if pd.isna(value) or not np.isfinite(value):
        return "nan"
    return f"{float(value):.4f}"


def evaluate(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'=' * 55}")
    print(f"  {model_name}")
    print(f"{'=' * 55}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R2    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "MAPE": round(mape, 2),
    }


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10, model_name="Model"):
    if not (0 < hotspot_percent <= 100):
        raise ValueError("hotspot_percent must be in (0, 100].")

    df_eval = pd.DataFrame(
        {
            "actual": np.asarray(y_true, dtype=float).reshape(-1),
            "predicted": np.asarray(y_pred, dtype=float).reshape(-1),
        }
    ).dropna().reset_index(drop=True)

    if df_eval.empty:
        return {
            "Model": model_name,
            "Hotspot_%": hotspot_percent,
            "Crimes_Captured": 0,
            "Total_Crimes": 0,
            "RRI": np.nan,
            "PAI": np.nan,
            "PEI": np.nan,
        }

    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    crimes_in_hotspots = float(df_eval.loc[df_eval["hotspot"], "actual"].sum())
    total_crimes = float(df_eval["actual"].sum())
    hotspot_area = int(df_eval["hotspot"].sum())
    total_area = int(len(df_eval))

    rri = safe_divide(crimes_in_hotspots, total_crimes)
    area_ratio = safe_divide(hotspot_area, total_area)
    pai = safe_divide(rri, area_ratio) if pd.notna(rri) and pd.notna(area_ratio) else np.nan
    pei = pai  # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Model             : {model_name}")
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {fmt4(rri)}")
    print(f"PAI               : {fmt4(pai)}")
    print(f"PEI               : {fmt4(pei)}")

    return {
        "Model": model_name,
        "Hotspot_%": hotspot_percent,
        "Crimes_Captured": int(round(crimes_in_hotspots)),
        "Total_Crimes": int(round(total_crimes)),
        "RRI": round_or_nan(rri, 4),
        "PAI": round_or_nan(pai, 4),
        "PEI": round_or_nan(pei, 4),
    }


def write_evaluation_report(
    report_path,
    sarima_result,
    hotspot_df,
    top_orders,
    fitted_lsoas,
    total_lsoas,
    skipped_count,
    failed_count,
    runtime_minutes,
    artifact_path,
):
    reg_out = pd.DataFrame([sarima_result])
    hs_cols = ["Model", "Hotspot_%", "Crimes_Captured", "Total_Crimes", "RRI", "PAI", "PEI"]
    hs_out = hotspot_df[hs_cols].copy().sort_values(["Model", "Hotspot_%"])

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# =============================================================================\n")
        f.write("# CRIME HOTSPOT EVALUATION METRICS\n")
        f.write("# PAI — Prediction Accuracy Index\n")
        f.write("# RRI — Recapture Rate Index\n")
        f.write("# PEI — Prediction Efficiency Index\n")
        f.write("# =============================================================================\n\n")
        f.write("Definitions\n")
        f.write("RRI = Crimes captured in hotspots / Total actual crimes\n")
        f.write("PAI = RRI / Hotspot area ratio\n")
        f.write("PEI = RRI / Hotspot area ratio (simplified approximation)\n\n")

        f.write("# =============================================================================\n")
        f.write("# SARIMA EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(reg_out[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))
        f.write("\n")
        f.write(f"LSOAs fitted    : {fitted_lsoas}\n")
        f.write(f"Total LSOAs     : {total_lsoas}\n")
        f.write(f"Skipped LSOAs   : {skipped_count}\n")
        f.write(f"Failed LSOAs    : {failed_count}\n")
        f.write(f"Runtime minutes : {runtime_minutes:.1f}\n\n")

        f.write("# =============================================================================\n")
        f.write("# HOTSPOT EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(hs_out.to_string(index=False))
        f.write("\n\n")

        f.write("# =============================================================================\n")
        f.write("# TOP ORDERS SELECTED\n")
        f.write("# =============================================================================\n")
        if top_orders:
            for order_key, count in top_orders:
                f.write(f"{order_key} : {count}\n")
        else:
            f.write("No order statistics available.\n")
        f.write("\n")

        f.write("# =============================================================================\n")
        f.write("# BEST MODEL EXPORT SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(f"Model selected : {sarima_result['Model']}\n")
        f.write("Metric used    : MAE\n")
        f.write(f"MAE            : {sarima_result['MAE']}\n")
        f.write(f"RMSE           : {sarima_result['RMSE']}\n")
        f.write(f"R2             : {sarima_result['R2']}\n")
        f.write(f"MAPE           : {sarima_result['MAPE']}\n")
        f.write(f"Artifact path  : {artifact_path}\n")


def try_import_auto_arima():
    try:
        from pmdarima import auto_arima as _auto_arima
        print("pmdarima available - using auto_arima for best order selection")
        return _auto_arima
    except Exception:
        print("pmdarima not found. Installing...")
        try:
            subprocess.run(
                ["pip", "install", "pmdarima", "--break-system-packages", "-q"],
                check=False,
            )
            from pmdarima import auto_arima as _auto_arima
            print("pmdarima installed successfully")
            return _auto_arima
        except Exception:
            print("pmdarima install failed - falling back to fixed order (1,1,1)(1,1,1,12)")
            return None


# =============================================================================
# 3. AUTO_ARIMA SETUP
# =============================================================================
auto_arima_fn = try_import_auto_arima()
AUTO_ARIMA_AVAILABLE = auto_arima_fn is not None

# =============================================================================
# 4. LOAD DATA + SPLIT
# =============================================================================
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["LSOA_Code", "Year", "Month"]).reset_index(drop=True)

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

train_df = df[df["Year"].isin(TRAIN_YEARS)].copy()
test_df = df[df["Year"] == TEST_YEAR].copy()
lsoa_list = df["LSOA_Code"].unique()

print(f"\nTotal LSOAs to fit : {len(lsoa_list)}")
print(f"Cores available    : {os.cpu_count()}")
print(f"Parallel jobs      : {'all cores' if N_JOBS == -1 else N_JOBS}")
print(f"Order selection    : {'auto_arima (AIC)' if AUTO_ARIMA_AVAILABLE else 'fixed (1,1,1)(1,1,1,12)'}")

# =============================================================================
# 5. STATIONARITY AUDIT (SAMPLE 10 LSOAs)
# =============================================================================
print("\n--- Stationarity audit (10 LSOAs) ---")
n_stationary = 0

for lsoa in lsoa_list[:10]:
    series = train_df.loc[train_df["LSOA_Code"] == lsoa, "Crime_Count"].values.astype(float)
    if len(series) < 4:
        continue
    p_val = adfuller(series, autolag="AIC")[1]
    status = "stationary" if p_val < 0.05 else "non-stationary"
    if p_val < 0.05:
        n_stationary += 1
    print(f"  {lsoa}  p={p_val:.4f}  {status}")

print(f"\n{n_stationary}/10 series are already stationary")
print("(non-stationary series will be corrected by differencing inside SARIMA)")

# =============================================================================
# 6. SINGLE LSOA FIT FUNCTION
# =============================================================================
def fit_sarima_lsoa(lsoa):
    try:
        lsoa_train = (
            train_df[train_df["LSOA_Code"] == lsoa]
            .sort_values(["Year", "Month"])["Crime_Count"]
            .values.astype(float)
        )

        lsoa_test = test_df[test_df["LSOA_Code"] == lsoa].sort_values("Month")

        if len(lsoa_train) < MIN_TRAIN_OBS:
            return {
                "lsoa": lsoa,
                "status": "skip_data",
                "rows": 0,
                "actuals": [],
                "forecasts": [],
                "months": [],
                "order_key": None,
            }

        if len(lsoa_test) == 0:
            return {
                "lsoa": lsoa,
                "status": "skip_notest",
                "rows": 0,
                "actuals": [],
                "forecasts": [],
                "months": [],
                "order_key": None,
            }

        if AUTO_ARIMA_AVAILABLE:
            try:
                auto_model = auto_arima_fn(lsoa_train, **AUTO_ARIMA_CONFIG)
                best_order = auto_model.order
                best_seasonal_order = auto_model.seasonal_order
            except Exception:
                best_order = FALLBACK_ORDER
                best_seasonal_order = FALLBACK_SEASONAL_ORDER
        else:
            best_order = FALLBACK_ORDER
            best_seasonal_order = FALLBACK_SEASONAL_ORDER

        model = SARIMAX(
            lsoa_train,
            order=best_order,
            seasonal_order=best_seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
            concentrate_scale=True,
        )

        fitted = model.fit(
            disp=False,
            method="lbfgs",
            maxiter=200,
            optim_score=None,
        )

        forecast = np.asarray(fitted.forecast(steps=N_FORECAST), dtype=float)
        forecast = np.nan_to_num(forecast, nan=0.0, posinf=0.0, neginf=0.0)
        forecast = np.clip(forecast, 0, None)

        actuals = []
        forecasts = []
        months = []

        for _, row in lsoa_test.iterrows():
            month = int(row["Month"])
            if 1 <= month <= N_FORECAST:
                actuals.append(float(row["Crime_Count"]))
                forecasts.append(float(forecast[month - 1]))
                months.append(month)

        return {
            "lsoa": lsoa,
            "status": "ok",
            "rows": len(actuals),
            "actuals": actuals,
            "forecasts": forecasts,
            "months": months,
            "order_key": f"{best_order} x {best_seasonal_order}",
            "aic": round(float(fitted.aic), 2),
        }

    except Exception as e:
        return {
            "lsoa": lsoa,
            "status": f"error: {str(e)[:80]}",
            "rows": 0,
            "actuals": [],
            "forecasts": [],
            "months": [],
            "order_key": None,
        }


# =============================================================================
# 7. PARALLEL FITTING
# =============================================================================
print(f"\nFitting SARIMA for all {len(lsoa_list)} LSOAs in parallel...")
print("This will take approximately 15-40 minutes depending on CPU count.\n")

start_time = time.time()
raw_results = Parallel(n_jobs=N_JOBS, verbose=5, backend="loky")(
    delayed(fit_sarima_lsoa)(lsoa) for lsoa in lsoa_list
)
elapsed = time.time() - start_time

print(f"\nParallel fitting complete in {elapsed / 60:.1f} minutes")

# =============================================================================
# 8. AGGREGATE RESULTS
# =============================================================================
all_actuals = []
all_forecasts = []
all_lsoas = []
all_months = []
orders_used = []
failed = []
skipped = []
fitted_lsoa_count = 0

for r in raw_results:
    if r["status"] == "ok" and r["rows"] > 0:
        fitted_lsoa_count += 1
        all_actuals.extend(r["actuals"])
        all_forecasts.extend(r["forecasts"])
        all_lsoas.extend([r["lsoa"]] * r["rows"])
        all_months.extend(r["months"])
        if r["order_key"] is not None:
            orders_used.append(r["order_key"])
    elif r["status"].startswith("skip"):
        skipped.append(r["lsoa"])
    else:
        failed.append((r["lsoa"], r["status"]))

print(f"\nSuccessfully predicted : {fitted_lsoa_count}")
print(f"Skipped (data issues)  : {len(skipped)}")
print(f"Failed (errors)        : {len(failed)}")
print(f"Total prediction rows  : {len(all_actuals):,}")

if failed:
    print("\nFirst 5 failures:")
    for lsoa, reason in failed[:5]:
        print(f"  {lsoa}: {reason}")

top_orders = Counter(orders_used).most_common(5)
if top_orders:
    print("\nTop 5 most common orders selected by auto_arima:")
    for order_key, count in top_orders:
        print(f"  {order_key} - {count} LSOAs")

if len(all_actuals) == 0:
    raise RuntimeError("No predictions were generated. Check data and configuration.")

# =============================================================================
# 9. EVALUATION + ERROR ANALYSIS
# =============================================================================
y_true = np.asarray(all_actuals, dtype=float)
y_pred = np.asarray(all_forecasts, dtype=float)

sarima_result = evaluate("SARIMA - auto order AIC (per LSOA)", y_true, y_pred)

print(f"  LSOAs fitted    : {fitted_lsoa_count}")
print(f"  Prediction rows : {len(y_true):,}")
print(f"  Runtime         : {elapsed / 60:.1f} minutes")

result_df = pd.DataFrame(
    {
        "LSOA_Code": all_lsoas,
        "Month": all_months,
        "Actual": y_true,
        "Predicted": np.round(y_pred, 2),
        "Abs_Error": np.abs(y_true - y_pred),
        "Pct_Error": np.abs(y_true - y_pred) / (y_true + 1e-9) * 100,
    }
)

result_df["Crime_Bucket"] = pd.cut(
    result_df["Actual"],
    bins=[0, 5, 15, 30, 50, 9999],
    labels=["0-5", "6-15", "16-30", "31-50", "50+"],
)

bucket_err = (
    result_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Avg_MAE", "count": "N_rows"})
)
print("\nError by crime count bucket:")
print(bucket_err.to_string())

monthly_err = (
    result_df.groupby("Month")["Abs_Error"]
    .mean()
    .reset_index()
    .rename(columns={"Abs_Error": "Avg_MAE"})
)
print("\nAverage error by month (1=Jan ... 12=Dec):")
print(monthly_err.to_string(index=False))

worst = (
    result_df.groupby("LSOA_Code")["Abs_Error"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop 10 hardest LSOAs:")
print(worst.to_string())

print("\n--- Sample Predictions vs Actual (first 12 rows) ---")
print(
    result_df.head(12)[
        ["LSOA_Code", "Month", "Actual", "Predicted", "Abs_Error", "Pct_Error"]
    ].to_string(index=False)
)

result_df.to_csv(PREDICTIONS_PATH, index=False)
print(f"\nDetailed predictions saved to {PREDICTIONS_PATH}")

# =============================================================================
# 10. SAVE TO MODEL COMPARISON CSV
# =============================================================================
new_row = pd.DataFrame([sarima_result])

try:
    existing = pd.read_csv(MODEL_COMPARISON_PATH)
    combined = pd.concat([existing, new_row], ignore_index=True)
except FileNotFoundError:
    combined = new_row

combined.to_csv(MODEL_COMPARISON_PATH, index=False)

print("\n--- Updated model_comparison.csv ---")
print(combined[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

# =============================================================================
# 11. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")

hotspot_results = []
for pct in HOTSPOT_PERCENTS:
    hs_res = crime_hotspot_metrics(
        y_true=y_true,
        y_pred=y_pred,
        hotspot_percent=pct,
        model_name=sarima_result["Model"],
    )
    hotspot_results.append(hs_res)

hotspot_df = pd.DataFrame(hotspot_results)

print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))

# =============================================================================
# 12. EXPORT BEST MODEL (SARIMA CONFIG ARTIFACT)
# =============================================================================
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

# SARIMA is fitted per LSOA at run time, so we export reproducibility metadata
# instead of serializing thousands of fitted state-space model objects.
best_model_package = {
    "model_name": sarima_result["Model"],
    "model_type": "SARIMA",
    "target": "Crime_Count",
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "n_forecast": N_FORECAST,
    "min_train_obs": MIN_TRAIN_OBS,
    "n_jobs": N_JOBS,
    "auto_arima_available": AUTO_ARIMA_AVAILABLE,
    "auto_arima_config": AUTO_ARIMA_CONFIG,
    "fallback_order": FALLBACK_ORDER,
    "fallback_seasonal_order": FALLBACK_SEASONAL_ORDER,
    "metrics": {
        "MAE": sarima_result["MAE"],
        "RMSE": sarima_result["RMSE"],
        "R2": sarima_result["R2"],
        "MAPE": sarima_result["MAPE"],
    },
    "fitted_lsoas": fitted_lsoa_count,
    "skipped_lsoas": len(skipped),
    "failed_lsoas": len(failed),
    "runtime_minutes": round(elapsed / 60, 2),
    "top_orders": top_orders,
    "hotspot_percents": HOTSPOT_PERCENTS,
    "predictions_file": PREDICTIONS_PATH,
}

joblib.dump(best_model_package, ARTIFACT_PATH)

print("\nBest model export complete")
print(f"Model selected : {sarima_result['Model']}")
print(f"Saved artifact : {ARTIFACT_PATH}")

# =============================================================================
# 13. WRITE SINGLE TEXT REPORT (OVERWRITE EACH RUN)
# =============================================================================
write_evaluation_report(
    report_path=REPORT_TXT_PATH,
    sarima_result=sarima_result,
    hotspot_df=hotspot_df,
    top_orders=top_orders,
    fitted_lsoas=fitted_lsoa_count,
    total_lsoas=len(lsoa_list),
    skipped_count=len(skipped),
    failed_count=len(failed),
    runtime_minutes=elapsed / 60,
    artifact_path=ARTIFACT_PATH,
)

print(f"\nEvaluation report saved to {REPORT_TXT_PATH}")

In [ ]:
# =============================================================================
# MODEL 5 - SARIMA (Optimised - full London, auto order selection)
# Task: Predict Crime_Count per LSOA based on historical time series
#
# Strategy:
#   - auto_arima selects best (p,d,q)(P,D,Q,12) per LSOA via AIC
#   - Parallel fitting across all CPU cores (joblib)
#   - Full London LSOA set - no sampling
#   - Refit on full 2020-2023 series, forecast all 12 months of 2024
# =============================================================================

import os
import time
import warnings
import subprocess
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================
DATA_PATH = "./crime_per_capita_df.csv"
MODEL_COMPARISON_PATH = "model_comparison.csv"
PREDICTIONS_PATH = "sarima_predictions.csv"
REPORT_TXT_PATH = "evaluation_results_sarima.txt"
ARTIFACT_PATH = Path("artifacts") / "sarima_model.joblib"

TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEAR = 2024
N_FORECAST = 12
MIN_TRAIN_OBS = 24
N_JOBS = -1

HOTSPOT_PERCENTS = [5, 10, 20]

SEASONAL_MIN_OBS = 48
LOW_VARIANCE_EPS = 1e-8
HIGH_ZERO_RATIO = 0.85
FORECAST_CAP_MULTIPLIER = 6.0
FORECAST_CAP_FLOOR = 80.0
GLOBAL_FORECAST_CAP_MULTIPLIER = 10.0

AUTO_ARIMA_CONFIG = {
    "start_p": 0,
    "max_p": 2,
    "start_q": 0,
    "max_q": 2,
    "d": None,
    "max_d": 2,
    "start_P": 0,
    "max_P": 1,
    "start_Q": 0,
    "max_Q": 1,
    "D": None,
    "max_D": 1,
    "m": 12,
    "seasonal": True,
    "information_criterion": "aic",
    "stepwise": True,
    "max_order": 5,
    "suppress_warnings": True,
    "error_action": "ignore",
    "trace": False,
}

FALLBACK_ORDER = (1, 1, 1)
FALLBACK_SEASONAL_ORDER = (1, 1, 1, 12)
SECONDARY_FALLBACK_ORDER = (1, 1, 0)
SECONDARY_FALLBACK_SEASONAL_ORDER = (0, 0, 0, 0)

# =============================================================================
# 2. HELPERS
# =============================================================================
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator == 0.0:
        return np.nan
    return float(numerator) / denominator


def round_or_nan(value, digits=4):
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), digits)


def fmt4(value):
    if pd.isna(value) or not np.isfinite(value):
        return "nan"
    return f"{float(value):.4f}"


def forecast_cap_from_train(train_values):
    if len(train_values) == 0:
        return FORECAST_CAP_FLOOR
    p99 = float(np.nanpercentile(train_values, 99))
    return max(FORECAST_CAP_FLOOR, p99 * FORECAST_CAP_MULTIPLIER)


def is_low_signal_series(train_values):
    if len(train_values) == 0:
        return True
    arr = np.asarray(train_values, dtype=float)
    zero_ratio = float(np.mean(arr <= 0.0))
    var_small = float(np.nanvar(arr)) < LOW_VARIANCE_EPS
    return var_small or zero_ratio >= HIGH_ZERO_RATIO


def sanitize_forecast(forecast_array, cap):
    forecast = np.asarray(forecast_array, dtype=float)
    forecast = np.nan_to_num(forecast, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(forecast, 0.0, cap)


def evaluate(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'=' * 55}")
    print(f"  {model_name}")
    print(f"{'=' * 55}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R2    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "MAPE": round(mape, 2),
    }


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI - Prediction Accuracy Index
# RRI - Recapture Rate Index
# PEI - Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10, model_name="Model"):
    if not (0 < hotspot_percent <= 100):
        raise ValueError("hotspot_percent must be in (0, 100].")

    df_eval = pd.DataFrame(
        {
            "actual": np.asarray(y_true, dtype=float).reshape(-1),
            "predicted": np.asarray(y_pred, dtype=float).reshape(-1),
        }
    ).dropna().reset_index(drop=True)

    if df_eval.empty:
        return {
            "Model": model_name,
            "Hotspot_%": hotspot_percent,
            "Crimes_Captured": 0,
            "Total_Crimes": 0,
            "RRI": np.nan,
            "PAI": np.nan,
            "PEI": np.nan,
        }

    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    crimes_in_hotspots = float(df_eval.loc[df_eval["hotspot"], "actual"].sum())
    total_crimes = float(df_eval["actual"].sum())
    hotspot_area = int(df_eval["hotspot"].sum())
    total_area = int(len(df_eval))

    rri = safe_divide(crimes_in_hotspots, total_crimes)
    area_ratio = safe_divide(hotspot_area, total_area)
    pai = safe_divide(rri, area_ratio) if pd.notna(rri) and pd.notna(area_ratio) else np.nan
    pei = pai

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Model             : {model_name}")
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {fmt4(rri)}")
    print(f"PAI               : {fmt4(pai)}")
    print(f"PEI               : {fmt4(pei)}")

    return {
        "Model": model_name,
        "Hotspot_%": hotspot_percent,
        "Crimes_Captured": int(round(crimes_in_hotspots)),
        "Total_Crimes": int(round(total_crimes)),
        "RRI": round_or_nan(rri, 4),
        "PAI": round_or_nan(pai, 4),
        "PEI": round_or_nan(pei, 4),
    }


def write_evaluation_report(
    report_path,
    sarima_result,
    hotspot_df,
    top_orders,
    fitted_lsoas,
    total_lsoas,
    skipped_count,
    failed_count,
    runtime_minutes,
    artifact_path,
):
    reg_out = pd.DataFrame([sarima_result])
    hs_cols = ["Model", "Hotspot_%", "Crimes_Captured", "Total_Crimes", "RRI", "PAI", "PEI"]
    hs_out = hotspot_df[hs_cols].copy().sort_values(["Model", "Hotspot_%"])

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# =============================================================================\n")
        f.write("# CRIME HOTSPOT EVALUATION METRICS\n")
        f.write("# PAI - Prediction Accuracy Index\n")
        f.write("# RRI - Recapture Rate Index\n")
        f.write("# PEI - Prediction Efficiency Index\n")
        f.write("# =============================================================================\n\n")
        f.write("Definitions\n")
        f.write("RRI = Crimes captured in hotspots / Total actual crimes\n")
        f.write("PAI = RRI / Hotspot area ratio\n")
        f.write("PEI = RRI / Hotspot area ratio (simplified approximation)\n\n")

        f.write("# =============================================================================\n")
        f.write("# SARIMA EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(reg_out[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))
        f.write("\n")
        f.write(f"LSOAs fitted    : {fitted_lsoas}\n")
        f.write(f"Total LSOAs     : {total_lsoas}\n")
        f.write(f"Skipped LSOAs   : {skipped_count}\n")
        f.write(f"Failed LSOAs    : {failed_count}\n")
        f.write(f"Runtime minutes : {runtime_minutes:.1f}\n\n")

        f.write("# =============================================================================\n")
        f.write("# HOTSPOT EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(hs_out.to_string(index=False))
        f.write("\n\n")

        f.write("# =============================================================================\n")
        f.write("# TOP ORDERS SELECTED\n")
        f.write("# =============================================================================\n")
        if top_orders:
            for order_key, count in top_orders:
                f.write(f"{order_key} : {count}\n")
        else:
            f.write("No order statistics available.\n")
        f.write("\n")

        f.write("# =============================================================================\n")
        f.write("# BEST MODEL EXPORT SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(f"Model selected : {sarima_result['Model']}\n")
        f.write("Metric used    : MAE\n")
        f.write(f"MAE            : {sarima_result['MAE']}\n")
        f.write(f"RMSE           : {sarima_result['RMSE']}\n")
        f.write(f"R2             : {sarima_result['R2']}\n")
        f.write(f"MAPE           : {sarima_result['MAPE']}\n")
        f.write(f"Artifact path  : {artifact_path}\n")


def try_import_auto_arima():
    try:
        from pmdarima import auto_arima as _auto_arima
        print("pmdarima available - using auto_arima for best order selection")
        return _auto_arima
    except Exception:
        print("pmdarima not found. Installing...")
        try:
            subprocess.run(
                ["pip", "install", "pmdarima", "--break-system-packages", "-q"],
                check=False,
            )
            from pmdarima import auto_arima as _auto_arima
            print("pmdarima installed successfully")
            return _auto_arima
        except Exception:
            print("pmdarima install failed - falling back to fixed order (1,1,1)(1,1,1,12)")
            return None


# =============================================================================
# 3. AUTO_ARIMA SETUP
# =============================================================================
auto_arima_fn = try_import_auto_arima()
AUTO_ARIMA_AVAILABLE = auto_arima_fn is not None

# =============================================================================
# 4. LOAD DATA + SPLIT
# =============================================================================
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["LSOA_Code", "Year", "Month"]).reset_index(drop=True)

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

train_df = df[df["Year"].isin(TRAIN_YEARS)].copy()
test_df = df[df["Year"] == TEST_YEAR].copy()
lsoa_list = df["LSOA_Code"].unique()

print(f"\nTotal LSOAs to fit : {len(lsoa_list)}")
print(f"Cores available    : {os.cpu_count()}")
print(f"Parallel jobs      : {'all cores' if N_JOBS == -1 else N_JOBS}")
print("Order selection    : adaptive auto_arima with robust fallback and forecast capping")

# =============================================================================
# 5. STATIONARITY AUDIT (SAMPLE 10 LSOAs)
# =============================================================================
print("\n--- Stationarity audit (10 LSOAs) ---")
n_stationary = 0

for lsoa in lsoa_list[:10]:
    series = train_df.loc[train_df["LSOA_Code"] == lsoa, "Crime_Count"].values.astype(float)
    if len(series) < 4:
        continue
    p_val = adfuller(series, autolag="AIC")[1]
    status = "stationary" if p_val < 0.05 else "non-stationary"
    if p_val < 0.05:
        n_stationary += 1
    print(f"  {lsoa}  p={p_val:.4f}  {status}")

print(f"\n{n_stationary}/10 series are already stationary")
print("(non-stationary series will be corrected by differencing inside SARIMA)")

# =============================================================================
# 6. SINGLE LSOA FIT FUNCTION
# =============================================================================
def fit_sarima_lsoa(lsoa):
    lsoa_train = np.array([], dtype=float)
    try:
        lsoa_train = (
            train_df[train_df["LSOA_Code"] == lsoa]
            .sort_values(["Year", "Month"])["Crime_Count"]
            .values.astype(float)
        )

        lsoa_test = test_df[test_df["LSOA_Code"] == lsoa].sort_values("Month")

        if len(lsoa_train) < MIN_TRAIN_OBS:
            return {
                "lsoa": lsoa,
                "status": "skip_data",
                "rows": 0,
                "actuals": [],
                "forecasts": [],
                "months": [],
                "order_key": None,
            }

        if len(lsoa_test) == 0:
            return {
                "lsoa": lsoa,
                "status": "skip_notest",
                "rows": 0,
                "actuals": [],
                "forecasts": [],
                "months": [],
                "order_key": None,
            }

        cap = forecast_cap_from_train(lsoa_train)
        low_signal = is_low_signal_series(lsoa_train)
        seasonal_allowed = (len(lsoa_train) >= SEASONAL_MIN_OBS) and (not low_signal)

        if low_signal:
            baseline = float(np.nanmedian(lsoa_train))
            forecast = np.full(N_FORECAST, max(0.0, baseline), dtype=float)
            best_order = (0, 0, 0)
            best_seasonal_order = (0, 0, 0, 0)
        else:
            if AUTO_ARIMA_AVAILABLE:
                try:
                    local_cfg = AUTO_ARIMA_CONFIG.copy()
                    local_cfg["seasonal"] = seasonal_allowed
                    local_cfg["m"] = 12 if seasonal_allowed else 1
                    if not seasonal_allowed:
                        local_cfg["start_P"] = 0
                        local_cfg["max_P"] = 0
                        local_cfg["start_Q"] = 0
                        local_cfg["max_Q"] = 0
                        local_cfg["max_D"] = 0

                    auto_model = auto_arima_fn(lsoa_train, **local_cfg)
                    best_order = auto_model.order
                    best_seasonal_order = auto_model.seasonal_order if seasonal_allowed else (0, 0, 0, 0)
                except Exception:
                    best_order = FALLBACK_ORDER
                    best_seasonal_order = FALLBACK_SEASONAL_ORDER if seasonal_allowed else (0, 0, 0, 0)
            else:
                best_order = FALLBACK_ORDER
                best_seasonal_order = FALLBACK_SEASONAL_ORDER if seasonal_allowed else (0, 0, 0, 0)

            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message="Too few observations to estimate starting parameters.*",
                )
                model = SARIMAX(
                    lsoa_train,
                    order=best_order,
                    seasonal_order=best_seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    concentrate_scale=True,
                )

                fitted = model.fit(
                    disp=False,
                    method="lbfgs",
                    maxiter=200,
                    optim_score=None,
                )

            converged = bool(getattr(fitted, "mle_retvals", {}).get("converged", True))
            if not converged:
                fallback_model = SARIMAX(
                    lsoa_train,
                    order=SECONDARY_FALLBACK_ORDER,
                    seasonal_order=SECONDARY_FALLBACK_SEASONAL_ORDER,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    concentrate_scale=True,
                )
                fitted = fallback_model.fit(
                    disp=False,
                    method="lbfgs",
                    maxiter=100,
                    optim_score=None,
                )
                best_order = SECONDARY_FALLBACK_ORDER
                best_seasonal_order = SECONDARY_FALLBACK_SEASONAL_ORDER

            forecast = sanitize_forecast(fitted.forecast(steps=N_FORECAST), cap)

        actuals = []
        forecasts = []
        months = []

        for _, row in lsoa_test.iterrows():
            month = int(row["Month"])
            if 1 <= month <= N_FORECAST:
                actuals.append(float(row["Crime_Count"]))
                forecasts.append(float(forecast[month - 1]))
                months.append(month)

        return {
            "lsoa": lsoa,
            "status": "ok",
            "rows": len(actuals),
            "actuals": actuals,
            "forecasts": forecasts,
            "months": months,
            "order_key": f"{best_order} x {best_seasonal_order}",
        }

    except Exception as e:
        if len(lsoa_train) > 0:
            lsoa_test = test_df[test_df["LSOA_Code"] == lsoa].sort_values("Month")
            cap = forecast_cap_from_train(lsoa_train)
            fallback_value = float(np.nanmean(lsoa_train[-12:]))
            fallback_value = min(max(0.0, fallback_value), cap)
            fallback_forecast = np.full(N_FORECAST, fallback_value, dtype=float)

            actuals = []
            forecasts = []
            months = []
            for _, row in lsoa_test.iterrows():
                month = int(row["Month"])
                if 1 <= month <= N_FORECAST:
                    actuals.append(float(row["Crime_Count"]))
                    forecasts.append(float(fallback_forecast[month - 1]))
                    months.append(month)

            return {
                "lsoa": lsoa,
                "status": "ok_fallback",
                "rows": len(actuals),
                "actuals": actuals,
                "forecasts": forecasts,
                "months": months,
                "order_key": "fallback_last12_mean",
            }

        return {
            "lsoa": lsoa,
            "status": f"error: {str(e)[:80]}",
            "rows": 0,
            "actuals": [],
            "forecasts": [],
            "months": [],
            "order_key": None,
        }


# =============================================================================
# 7. PARALLEL FITTING
# =============================================================================
print(f"\nFitting SARIMA for all {len(lsoa_list)} LSOAs in parallel...")
print("This will take approximately 15-40 minutes depending on CPU count.\n")

start_time = time.time()
raw_results = Parallel(n_jobs=N_JOBS, verbose=5, backend="loky")(
    delayed(fit_sarima_lsoa)(lsoa) for lsoa in lsoa_list
)
elapsed = time.time() - start_time

print(f"\nParallel fitting complete in {elapsed / 60:.1f} minutes")

# =============================================================================
# 8. AGGREGATE RESULTS
# =============================================================================
all_actuals = []
all_forecasts = []
all_lsoas = []
all_months = []
orders_used = []
failed = []
skipped = []
fitted_lsoa_count = 0

for r in raw_results:
    if r["status"].startswith("ok") and r["rows"] > 0:
        fitted_lsoa_count += 1
        all_actuals.extend(r["actuals"])
        all_forecasts.extend(r["forecasts"])
        all_lsoas.extend([r["lsoa"]] * r["rows"])
        all_months.extend(r["months"])
        if r["order_key"] is not None:
            orders_used.append(r["order_key"])
    elif r["status"].startswith("skip"):
        skipped.append(r["lsoa"])
    else:
        failed.append((r["lsoa"], r["status"]))

print(f"\nSuccessfully predicted : {fitted_lsoa_count}")
print(f"Skipped (data issues)  : {len(skipped)}")
print(f"Failed (errors)        : {len(failed)}")
print(f"Total prediction rows  : {len(all_actuals):,}")

if failed:
    print("\nFirst 5 failures:")
    for lsoa, reason in failed[:5]:
        print(f"  {lsoa}: {reason}")

top_orders = Counter(orders_used).most_common(5)
if top_orders:
    print("\nTop 5 most common orders selected by auto_arima:")
    for order_key, count in top_orders:
        print(f"  {order_key} - {count} LSOAs")

if len(all_actuals) == 0:
    raise RuntimeError("No predictions were generated. Check data and configuration.")

# =============================================================================
# 9. EVALUATION + ERROR ANALYSIS
# =============================================================================
y_true = np.asarray(all_actuals, dtype=float)
y_pred = np.asarray(all_forecasts, dtype=float)

y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
global_cap = max(
    FORECAST_CAP_FLOOR,
    float(np.nanpercentile(y_true, 99.9)) * GLOBAL_FORECAST_CAP_MULTIPLIER,
)
n_global_clipped = int(np.sum(y_pred > global_cap))
if n_global_clipped > 0:
    print(f"Clipping {n_global_clipped:,} predictions above global cap {global_cap:.2f}")
    y_pred = np.clip(y_pred, 0.0, global_cap)

sarima_result = evaluate("SARIMA - auto order AIC (per LSOA)", y_true, y_pred)

print(f"  LSOAs fitted    : {fitted_lsoa_count}")
print(f"  Prediction rows : {len(y_true):,}")
print(f"  Runtime         : {elapsed / 60:.1f} minutes")

result_df = pd.DataFrame(
    {
        "LSOA_Code": all_lsoas,
        "Month": all_months,
        "Actual": y_true,
        "Predicted": np.round(y_pred, 2),
        "Abs_Error": np.abs(y_true - y_pred),
        "Pct_Error": np.abs(y_true - y_pred) / (y_true + 1e-9) * 100,
    }
)

result_df["Crime_Bucket"] = pd.cut(
    result_df["Actual"],
    bins=[0, 5, 15, 30, 50, 9999],
    labels=["0-5", "6-15", "16-30", "31-50", "50+"],
)

bucket_err = (
    result_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Avg_MAE", "count": "N_rows"})
)
print("\nError by crime count bucket:")
print(bucket_err.to_string())

monthly_err = (
    result_df.groupby("Month")["Abs_Error"]
    .mean()
    .reset_index()
    .rename(columns={"Abs_Error": "Avg_MAE"})
)
print("\nAverage error by month (1=Jan ... 12=Dec):")
print(monthly_err.to_string(index=False))

worst = (
    result_df.groupby("LSOA_Code")["Abs_Error"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop 10 hardest LSOAs:")
print(worst.to_string())

print("\n--- Sample Predictions vs Actual (first 12 rows) ---")
print(
    result_df.head(12)[
        ["LSOA_Code", "Month", "Actual", "Predicted", "Abs_Error", "Pct_Error"]
    ].to_string(index=False)
)

result_df.to_csv(PREDICTIONS_PATH, index=False)
print(f"\nDetailed predictions saved to {PREDICTIONS_PATH}")

# =============================================================================
# 10. SAVE TO MODEL COMPARISON CSV
# =============================================================================
new_row = pd.DataFrame([sarima_result])

try:
    existing = pd.read_csv(MODEL_COMPARISON_PATH)
    combined = pd.concat([existing, new_row], ignore_index=True)
except FileNotFoundError:
    combined = new_row

combined.to_csv(MODEL_COMPARISON_PATH, index=False)

print("\n--- Updated model_comparison.csv ---")
print(combined[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

# =============================================================================
# 11. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")

hotspot_results = []
for pct in HOTSPOT_PERCENTS:
    hs_res = crime_hotspot_metrics(
        y_true=y_true,
        y_pred=y_pred,
        hotspot_percent=pct,
        model_name=sarima_result["Model"],
    )
    hotspot_results.append(hs_res)

hotspot_df = pd.DataFrame(hotspot_results)

print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))

# =============================================================================
# 12. EXPORT BEST MODEL (SARIMA CONFIG ARTIFACT)
# =============================================================================
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

# SARIMA is fitted per LSOA at run time, so we export reproducibility metadata
# instead of serializing thousands of fitted state-space model objects.
best_model_package = {
    "model_name": sarima_result["Model"],
    "model_type": "SARIMA",
    "target": "Crime_Count",
    "train_years": TRAIN_YEARS,
    "test_year": TEST_YEAR,
    "n_forecast": N_FORECAST,
    "min_train_obs": MIN_TRAIN_OBS,
    "n_jobs": N_JOBS,
    "auto_arima_available": AUTO_ARIMA_AVAILABLE,
    "auto_arima_config": AUTO_ARIMA_CONFIG,
    "fallback_order": FALLBACK_ORDER,
    "fallback_seasonal_order": FALLBACK_SEASONAL_ORDER,
    "secondary_fallback_order": SECONDARY_FALLBACK_ORDER,
    "secondary_fallback_seasonal_order": SECONDARY_FALLBACK_SEASONAL_ORDER,
    "seasonal_min_obs": SEASONAL_MIN_OBS,
    "low_variance_eps": LOW_VARIANCE_EPS,
    "high_zero_ratio": HIGH_ZERO_RATIO,
    "forecast_cap_multiplier": FORECAST_CAP_MULTIPLIER,
    "forecast_cap_floor": FORECAST_CAP_FLOOR,
    "global_forecast_cap_multiplier": GLOBAL_FORECAST_CAP_MULTIPLIER,
    "global_forecast_cap": round(float(global_cap), 4),
    "metrics": {
        "MAE": sarima_result["MAE"],
        "RMSE": sarima_result["RMSE"],
        "R2": sarima_result["R2"],
        "MAPE": sarima_result["MAPE"],
    },
    "fitted_lsoas": fitted_lsoa_count,
    "skipped_lsoas": len(skipped),
    "failed_lsoas": len(failed),
    "runtime_minutes": round(elapsed / 60, 2),
    "top_orders": top_orders,
    "hotspot_percents": HOTSPOT_PERCENTS,
    "predictions_file": PREDICTIONS_PATH,
}

joblib.dump(best_model_package, ARTIFACT_PATH)

print("\nBest model export complete")
print(f"Model selected : {sarima_result['Model']}")
print(f"Saved artifact : {ARTIFACT_PATH}")

# =============================================================================
# 13. WRITE SINGLE TEXT REPORT (OVERWRITE EACH RUN)
# =============================================================================
write_evaluation_report(
    report_path=REPORT_TXT_PATH,
    sarima_result=sarima_result,
    hotspot_df=hotspot_df,
    top_orders=top_orders,
    fitted_lsoas=fitted_lsoa_count,
    total_lsoas=len(lsoa_list),
    skipped_count=len(skipped),
    failed_count=len(failed),
    runtime_minutes=elapsed / 60,
    artifact_path=ARTIFACT_PATH,
)

print(f"\nEvaluation report saved to {REPORT_TXT_PATH}")

pmdarima available - using auto_arima for best order selection

Dataset shape : (304336, 39)
Date range    : 2020 to 2024
Unique LSOAs  : 12415

Total LSOAs to fit : 12415
Cores available    : 4
Parallel jobs      : all cores
Order selection    : adaptive auto_arima with robust fallback and forecast capping

--- Stationarity audit (10 LSOAs) ---
  E01000001  p=0.0000  stationary
  E01000002  p=0.0002  stationary
  E01000003  p=0.9789  non-stationary
  E01000005  p=0.8235  non-stationary
  E01000006  p=0.0000  stationary
  E01000007  p=0.0000  stationary
  E01000008  p=0.0001  stationary
  E01000009  p=0.7702  non-stationary
  E01000011  p=0.7502  non-stationary
  E01000012  p=0.0002  stationary

6/10 series are already stationary
(non-stationary series will be corrected by differencing inside SARIMA)

Fitting SARIMA for all 12415 LSOAs in parallel...
This will take approximately 15-40 minutes depending on CPU count.



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    9.4s
/workspaces/Predictive-Policing-in-the-London-Metropolitan-Police-District/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/kalman_filter.py:1022: RuntimeWarning: invalid value encountered in scalar divide
  scale = np.sum(kfilter.scale[d:]) / nobs_k_endog
/workspaces/Predictive-Policing-in-the-London-Metropolitan-Police-District/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:   29.8s
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  1.1min
/workspaces/Predictive-Policing-in-the-London-Metropolitan-Police-District/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelih